# 03. Experience Replay

Hai khung hình liền nhau gần như giống nhau. Nếu mạng học theo đúng thứ tự chơi, nó sẽ khớp đoạn đang chơi và quên các đoạn trước. Experience Replay lưu nhiều transition gần đây và mỗi lần học lấy ngẫu nhiên một batch trong đó. Cách này dùng được vì Q-Learning là off-policy.

Nội dung:

1. Lớp `ReplayBuffer`.
2. Kiểm tra nhanh.

## 1. Lớp `ReplayBuffer`

Mỗi transition gồm `(s, a, r, s', terminated)`. Cờ `terminated` cho biết chim đã va chạm, khi đó Target chỉ bằng reward. Bước bị cắt vì `score_limit` (`truncated`) được lưu với `terminated = False`, vì chim chưa chết.

- Bộ nhớ được cấp sẵn bằng các mảng numpy kích thước cố định. Khi đầy, transition mới ghi đè transition cũ nhất (bộ đệm vòng).
- Với sức chứa 100.000 transition, các mảng `float32` chiếm khoảng 11 MB.
- `sample()` lấy ngẫu nhiên có hoàn lại; với 100.000 phần tử và batch 64, việc trùng mẫu trong cùng batch rất hiếm.

In [ ]:
import numpy as np


class ReplayBuffer:
    def __init__(self, capacity, obs_dim=12, seed=None):
        self.capacity = capacity
        self._states = np.zeros((capacity, obs_dim), dtype=np.float32)
        self._actions = np.zeros(capacity, dtype=np.int32)
        self._rewards = np.zeros(capacity, dtype=np.float32)
        self._next_states = np.zeros((capacity, obs_dim), dtype=np.float32)
        self._terminated = np.zeros(capacity, dtype=np.float32)
        self._next_index = 0
        self._size = 0
        self._rng = np.random.default_rng(seed)

    def __len__(self):
        return self._size

    def add(self, state, action, reward, next_state, terminated):
        i = self._next_index
        self._states[i] = state
        self._actions[i] = action
        self._rewards[i] = reward
        self._next_states[i] = next_state
        self._terminated[i] = float(terminated)
        self._next_index = (i + 1) % self.capacity
        self._size = min(self._size + 1, self.capacity)

    def sample(self, batch_size):
        if batch_size > self._size:
            raise ValueError(f"Cần ít nhất {batch_size} transition, mới có {self._size}.")
        idx = self._rng.integers(0, self._size, size=batch_size)
        return (self._states[idx], self._actions[idx], self._rewards[idx],
                self._next_states[idx], self._terminated[idx])

## 2. Kiểm tra nhanh

- Kích thước không vượt sức chứa, và khi đầy thì transition cũ nhất bị ghi đè.
- `sample()` trả về đúng dạng và báo lỗi khi chưa đủ dữ liệu.

In [ ]:
def _self_check_replay_buffer():
    buffer = ReplayBuffer(capacity=3, obs_dim=12, seed=0)
    try:
        buffer.sample(1)
        raise AssertionError("sample() khi rỗng phải báo lỗi")
    except ValueError:
        pass

    for k in range(5):
        state = np.full(12, k, dtype=np.float32)
        buffer.add(state, k % 2, 0.1 * k, state + 1, terminated=(k == 4))
    assert len(buffer) == 3
    # Transition 0 và 1 đã bị ghi đè, còn lại 2, 3, 4
    assert set(buffer._states[:, 0].tolist()) == {2.0, 3.0, 4.0}

    states, actions, rewards, next_states, terminated = buffer.sample(8)
    assert states.shape == (8, 12) and next_states.shape == (8, 12)
    assert actions.shape == rewards.shape == terminated.shape == (8,)
    assert np.allclose(next_states, states + 1)
    print("03_replay_buffer: kiểm tra đạt.")


_self_check_replay_buffer()